# TalkDB quickstart

Five-minute walkthrough: seed a demo ecommerce DB, build the hybrid retrieval index, and ask two questions in plain English. You get back validated SQL, the rows, and a confidence score.

**Prereq:** `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` in your `.env`. See [.env.example](../.env.example).

In [1]:
import os
from pathlib import Path

# Locate project root (so relative paths data/, packages/, .env resolve regardless of
# where the notebook was launched from).
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('could not locate project root (no pyproject.toml found upward)')
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv()
if not (os.environ.get('OPENAI_API_KEY') or os.environ.get('ANTHROPIC_API_KEY')):
    raise RuntimeError('Set OPENAI_API_KEY or ANTHROPIC_API_KEY in your .env before running this notebook.')
print('project root:', PROJECT_ROOT)

project root: /Users/nitingupta/Desktop/talkdb


## Seed the demo database

Creates `data/example.db` with 200 customers, 50 products, ~800 orders.

In [2]:
import subprocess, sys

subprocess.run([sys.executable, 'scripts/seed_example_db.py'], check=True)

Seeded data/example.db
  customers: 200
  products:  50
  orders:    786
  order_items: 1926


CompletedProcess(args=['/Users/nitingupta/Desktop/talkdb/.venv/bin/python3.14', 'scripts/seed_example_db.py'], returncode=0)

## Build the engine + index

In [3]:
from talkdb.config.settings import get_settings
from talkdb.core.engine import Engine

engine = Engine(get_settings())
doc_count = engine.build_index()
print(f'Indexed {doc_count} documents into the hybrid retriever.')

Indexed 36 documents into the hybrid retriever.


## Ask a question

In [4]:
result = await engine.ask('How many customers are there?')
print('SQL:       ', result.sql)
print('Rows:      ', result.results)
print('Confidence:', result.confidence)

SQL:        SELECT COUNT(*) FROM customers
Rows:       [{'COUNT(*)': 200}]
Confidence: 73


## A harder aggregation — joining three tables

Revenue by product category, completed orders only.

In [5]:
result = await engine.ask(
    'Which product category has the highest total revenue from completed orders?'
)
print('SQL:')
print(result.sql)
print()
print('Top rows:')
for row in result.results[:5]:
    print(' ', row)
print()
print('Confidence:', result.confidence)

SQL:
SELECT p.category, SUM(oi.quantity * oi.unit_price) AS revenue FROM order_items oi JOIN orders o ON oi.order_id = o.id JOIN products p ON oi.product_id = p.id WHERE o.status = 'completed' GROUP BY p.category ORDER BY revenue DESC LIMIT 1

Top rows:
  {'category': 'electronics', 'revenue': 223661.44}

Confidence: 75
